# Notebook 07 — Cell-Type PD Analysis (GSE184950)

## Objetivo
Realizar análisis transcriptómico por **tipo celular** en sustancia negra usando **pseudobulk por donante × tipo celular**.

### Principios estadísticos
- La unidad biológica es el **donante**, no el núcleo.
- Los conteos crudos se leen exclusivamente desde `adata.layers["counts"]`.
- Las múltiples muestras/GSM del mismo donante se agregan juntas.
- Comparación primaria: **PD vs Control**.
- PDD se conserva para análisis secundario.
- Sólo se analizan tipos celulares con representación suficiente de donantes en ambos grupos.
- La firma metabólica del Notebook 05 se evalúa explícitamente por tipo celular.

## 0. Imports, configuración y rutas

In [ ]:
from pathlib import Path
import json, gc, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

RANDOM_STATE = 42
ALPHA = 0.05
cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"C:\Users\Camfe\pythonlab")
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
GSE_DIR = DATA_DIR / "GSE184950"
PROCESSED_DIR = GSE_DIR / "processed"
NB07_RESULTS = RESULTS_DIR / "single_cell" / "notebook07"
NB07_FIGURES = FIGURES_DIR / "single_cell" / "notebook07"
NB07_RESULTS.mkdir(parents=True, exist_ok=True)
NB07_FIGURES.mkdir(parents=True, exist_ok=True)
H5AD_PATH = PROCESSED_DIR / "GSE184950_preprocessed_QC.h5ad"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("H5AD:", H5AD_PATH)
print("Exists:", H5AD_PATH.exists())

## 1. Dependencias

In [ ]:
try:
    import scanpy as sc
    import anndata as ad
    print("scanpy:", sc.__version__)
    print("anndata:", ad.__version__)
except Exception as exc:
    raise ImportError("Notebook 07 requires scanpy and anndata") from exc
try:
    import pydeseq2
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
    print("pydeseq2:", pydeseq2.__version__)
    HAS_PYDESEQ2 = True
except Exception as exc:
    HAS_PYDESEQ2 = False
    print("PyDESeq2 unavailable:", exc)
print("PyDESeq2 available:", HAS_PYDESEQ2)

## 2. Cargar AnnData procesado

In [ ]:
if not H5AD_PATH.exists():
    raise FileNotFoundError(f"Processed AnnData not found:\n{H5AD_PATH}")
adata = sc.read_h5ad(H5AD_PATH)
print(adata)
print("Shape:", adata.shape)
print("Raw counts layer:", "counts" in adata.layers)
print("adata.raw:", adata.raw is not None)

## 3. Validaciones críticas de integridad

In [ ]:
required_obs = ["donor_id", "disease", "cell_type_provisional"]
missing_obs = [c for c in required_obs if c not in adata.obs.columns]
if missing_obs:
    raise KeyError(f"Missing required obs columns: {missing_obs}")
if "counts" not in adata.layers:
    raise KeyError("Raw count layer adata.layers['counts'] is missing")
if not adata.obs_names.is_unique:
    raise ValueError("Cell/nucleus IDs are not unique")
counts_layer = adata.layers["counts"]
print("Counts sparse:", sparse.issparse(counts_layer))
print("Counts dtype:", counts_layer.dtype)
if sparse.issparse(counts_layer): print("Counts nnz:", f"{counts_layer.nnz:,}")
display(adata.obs["disease"].value_counts(dropna=False))
display(adata.obs["cell_type_provisional"].value_counts(dropna=False))

## 4. Estandarizar etiquetas

In [ ]:
adata.obs["donor_id_clean"] = adata.obs["donor_id"].astype(str).str.strip()
adata.obs["disease_clean"] = adata.obs["disease"].astype(str).str.strip()
adata.obs["cell_type_clean"] = adata.obs["cell_type_provisional"].astype(str).str.strip()
print("Disease:", sorted(adata.obs["disease_clean"].unique()))
print("Donors:", adata.obs["donor_id_clean"].nunique())
print("Cell types:", adata.obs["cell_type_clean"].nunique())

## 5. Auditoría de donantes y muestras repetidas

In [ ]:
group_cols=["donor_id_clean","disease_clean"]
donor_summary=(adata.obs.groupby(group_cols,observed=True).agg(n_nuclei=("donor_id_clean","size"),n_cell_types=("cell_type_clean","nunique")).reset_index())
if "geo_accession" in adata.obs.columns:
    gsm_counts=(adata.obs.groupby(group_cols,observed=True)["geo_accession"].nunique().rename("n_GSM").reset_index())
    donor_summary=donor_summary.merge(gsm_counts,on=group_cols,how="left")
donor_summary.to_csv(NB07_RESULTS/"GSE184950_donor_summary.csv",index=False)
display(donor_summary.sort_values(["disease_clean","donor_id_clean"]))

## 6. Verificar que un donante no aparezca en más de una enfermedad

In [ ]:
donor_disease_check=adata.obs.groupby("donor_id_clean",observed=True)["disease_clean"].nunique()
conflicted_donors=donor_disease_check[donor_disease_check>1]
print("Donors associated with >1 disease label:",len(conflicted_donors))
if len(conflicted_donors)>0:
    display(conflicted_donors)
    raise ValueError("At least one donor has multiple disease labels")

## 7. Cobertura donante × tipo celular

In [ ]:
coverage=(adata.obs.groupby(["donor_id_clean","disease_clean","cell_type_clean"],observed=True).size().rename("n_nuclei").reset_index())
coverage.to_csv(NB07_RESULTS/"GSE184950_donor_celltype_coverage.csv",index=False)
display(coverage.sort_values(["cell_type_clean","disease_clean","donor_id_clean"]).head(50))

## 8. Umbrales de representación

Criterio inicial:
- ≥20 núcleos por pseudobulk donante×tipo celular.
- ≥3 donantes Control y ≥3 donantes PD por tipo celular.

`3 vs 3` es un mínimo técnico, no implica alta potencia estadística.

In [ ]:
MIN_NUCLEI_PER_PSEUDOBULK=20
MIN_DONORS_PER_GROUP=3
coverage["eligible_pseudobulk"]=coverage["n_nuclei"]>=MIN_NUCLEI_PER_PSEUDOBULK
eligible_cov=coverage.loc[coverage["eligible_pseudobulk"]].copy()
representation=(eligible_cov.groupby(["cell_type_clean","disease_clean"],observed=True)["donor_id_clean"].nunique().rename("n_donors").reset_index())
rep_pivot=(representation.pivot(index="cell_type_clean",columns="disease_clean",values="n_donors").fillna(0).astype(int))
for group in ["Control","PD","PDD"]:
    if group not in rep_pivot.columns: rep_pivot[group]=0
rep_pivot["eligible_primary_DE"]=(rep_pivot["Control"]>=MIN_DONORS_PER_GROUP)&(rep_pivot["PD"]>=MIN_DONORS_PER_GROUP)
rep_pivot=rep_pivot.sort_values(["eligible_primary_DE","Control","PD"],ascending=[False,False,False])
rep_pivot.to_csv(NB07_RESULTS/"GSE184950_celltype_representation.csv")
display(rep_pivot)

## 9. Tipos celulares elegibles para PD vs Control

In [ ]:
eligible_celltypes=rep_pivot.index[rep_pivot["eligible_primary_DE"]].astype(str).tolist()
print(f"Eligible cell types ({len(eligible_celltypes)}):")
for ct in eligible_celltypes:
    print(f"- {ct}: Control={rep_pivot.loc[ct,'Control']}, PD={rep_pivot.loc[ct,'PD']}")
if len(eligible_celltypes)==0:
    raise RuntimeError("No cell type has enough Control and PD donors")

## 10. Evaluación de covariables

Se revisan edad, sexo y PMI. El modelo primario permanece `~ disease`. Braak no se usa como covariable rutinaria por su relación con severidad/enfermedad.

In [ ]:
candidate_covariates=[c for c in ["age","sex","PMI_hours"] if c in adata.obs.columns]
donor_cols=["donor_id_clean","disease_clean"]+candidate_covariates
donor_metadata=adata.obs[donor_cols].drop_duplicates().copy()
for cov in candidate_covariates:
    nvals=donor_metadata.groupby("donor_id_clean",observed=True)[cov].nunique(dropna=True)
    bad=nvals[nvals>1]
    if len(bad): warnings.warn(f"{cov}: {len(bad)} donors have multiple values")
donor_metadata=(donor_metadata.groupby(["donor_id_clean","disease_clean"],observed=True,as_index=False).first())
donor_metadata.to_csv(NB07_RESULTS/"GSE184950_donor_covariates.csv",index=False)
display(donor_metadata)
display(donor_metadata.isna().mean().sort_values(ascending=False))

## 11. Visualizar covariables por grupo

In [ ]:
for cov in ["age","PMI_hours"]:
    if cov not in donor_metadata.columns: continue
    tmp=donor_metadata[donor_metadata["disease_clean"].isin(["Control","PD"])].copy()
    tmp[cov]=pd.to_numeric(tmp[cov],errors="coerce"); tmp=tmp.dropna(subset=[cov])
    groups=[]; labels=[]
    for disease in ["Control","PD"]:
        values=tmp.loc[tmp["disease_clean"]==disease,cov].to_numpy(dtype=float)
        if len(values): groups.append(values); labels.append(disease)
    if len(groups)>=2:
        fig,ax=plt.subplots(figsize=(5,4)); ax.boxplot(groups,labels=labels); ax.set_ylabel(cov); ax.set_title(f"{cov} by disease")
        fig.tight_layout(); fig.savefig(NB07_FIGURES/f"covariate_{cov}_by_disease.png",dpi=300,bbox_inches="tight"); plt.show(); plt.close(fig)

## 12. Función de agregación pseudobulk

In [ ]:
def aggregate_pseudobulk(adata,cell_type,min_nuclei=20,diseases=("Control","PD")):
    obs=adata.obs
    mask=obs["cell_type_clean"].eq(cell_type)&obs["disease_clean"].isin(diseases)
    idx=np.flatnonzero(mask.to_numpy())
    if len(idx)==0: return None,None,None
    sub_obs=obs.iloc[idx].copy()
    donor_cov=(sub_obs.groupby(["donor_id_clean","disease_clean"],observed=True).size().rename("n_nuclei").reset_index())
    keep_donors=donor_cov.loc[donor_cov["n_nuclei"]>=min_nuclei,"donor_id_clean"].astype(str).tolist()
    idx=np.flatnonzero((obs["cell_type_clean"].eq(cell_type)&obs["disease_clean"].isin(diseases)&obs["donor_id_clean"].isin(keep_donors)).to_numpy())
    if len(idx)==0: return None,None,donor_cov
    X=adata.layers["counts"][idx,:]
    X=X.tocsr() if sparse.issparse(X) else sparse.csr_matrix(X)
    local_obs=obs.iloc[idx]
    donors=sorted(local_obs["donor_id_clean"].astype(str).unique())
    rows=[]; meta_rows=[]
    for donor in donors:
        donor_positions=np.flatnonzero(local_obs["donor_id_clean"].astype(str).eq(donor).to_numpy())
        summed=np.asarray(X[donor_positions,:].sum(axis=0)).ravel()
        summed=np.rint(summed).astype(np.int64)
        disease_values=local_obs.iloc[donor_positions]["disease_clean"].astype(str).unique()
        if len(disease_values)!=1: raise ValueError(f"Donor {donor} has inconsistent disease labels")
        rows.append(summed)
        row={"sample_id":f"{cell_type}__{donor}","donor_id":donor,"disease":disease_values[0],"cell_type":cell_type,"n_nuclei":int(len(donor_positions))}
        for cov in ["age","sex","PMI_hours"]:
            if cov in local_obs.columns:
                vals=local_obs.iloc[donor_positions][cov].dropna().unique(); row[cov]=vals[0] if len(vals) else np.nan
        meta_rows.append(row)
    counts_df=pd.DataFrame(np.vstack(rows),index=[r["sample_id"] for r in meta_rows],columns=adata.var_names.astype(str))
    metadata_df=pd.DataFrame(meta_rows).set_index("sample_id")
    return counts_df,metadata_df,donor_cov

## 13. Construir y auditar pseudobulk

In [ ]:
pseudobulk_objects={}; pseudobulk_summary_rows=[]
for cell_type in eligible_celltypes:
    print("\n"+"="*70); print("Cell type:",cell_type)
    counts_df,metadata_df,donor_cov=aggregate_pseudobulk(adata,cell_type,MIN_NUCLEI_PER_PSEUDOBULK,("Control","PD"))
    if counts_df is None: print("No usable pseudobulk"); continue
    n_control=int((metadata_df["disease"]=="Control").sum()); n_pd=int((metadata_df["disease"]=="PD").sum())
    print("Pseudobulk samples:",len(metadata_df),"Control:",n_control,"PD:",n_pd,"Genes:",counts_df.shape[1])
    assert (counts_df.values>=0).all(); assert np.allclose(counts_df.values,np.rint(counts_df.values))
    pseudobulk_objects[cell_type]={"counts":counts_df,"metadata":metadata_df}
    pseudobulk_summary_rows.append({"cell_type":cell_type,"n_control":n_control,"n_PD":n_pd,"n_total_donors":len(metadata_df),"min_nuclei":int(metadata_df["n_nuclei"].min()),"median_nuclei":float(metadata_df["n_nuclei"].median()),"max_nuclei":int(metadata_df["n_nuclei"].max()),"n_genes_before_filter":counts_df.shape[1]})
pseudobulk_summary=pd.DataFrame(pseudobulk_summary_rows)
pseudobulk_summary.to_csv(NB07_RESULTS/"GSE184950_pseudobulk_summary.csv",index=False)
display(pseudobulk_summary)

## 14. Filtro de genes para DESeq2

In [ ]:
MIN_COUNT=10; MIN_SAMPLES_WITH_COUNT=3
filtered_pseudobulk_objects={}
for cell_type,obj in pseudobulk_objects.items():
    counts_df=obj["counts"]; metadata_df=obj["metadata"]
    keep=(counts_df>=MIN_COUNT).sum(axis=0)>=MIN_SAMPLES_WITH_COUNT
    filtered=counts_df.loc[:,keep].copy()
    filtered_pseudobulk_objects[cell_type]={"counts":filtered,"metadata":metadata_df.copy()}
    print(f"{cell_type}: {counts_df.shape[1]:,} -> {filtered.shape[1]:,} genes")

## 15. Tamaños de biblioteca pseudobulk

In [ ]:
library_rows=[]
for cell_type,obj in filtered_pseudobulk_objects.items():
    counts_df=obj["counts"]; metadata_df=obj["metadata"]; lib=counts_df.sum(axis=1)
    tmp=metadata_df.copy(); tmp["library_size"]=lib; tmp["cell_type"]=cell_type; library_rows.append(tmp.reset_index())
pseudobulk_libraries=pd.concat(library_rows,ignore_index=True) if library_rows else pd.DataFrame()
pseudobulk_libraries.to_csv(NB07_RESULTS/"GSE184950_pseudobulk_library_sizes.csv",index=False)
display(pseudobulk_libraries)

## 16. Función PyDESeq2

Modelo primario `~ disease`; contraste `PD vs Control`. LFC positivo = mayor expresión en PD.

In [ ]:
def run_pydeseq2_primary(counts_df,metadata_df,alpha=0.05):
    if not HAS_PYDESEQ2: raise ImportError("PyDESeq2 is not installed")
    counts=counts_df.astype(np.int64).copy(); metadata=metadata_df.copy(); assert counts.index.equals(metadata.index)
    metadata["disease"]=pd.Categorical(metadata["disease"],categories=["Control","PD"])
    dds=DeseqDataSet(counts=counts,metadata=metadata,design="~ disease",refit_cooks=True,low_memory=True,quiet=False)
    dds.deseq2()
    ds=DeseqStats(dds,contrast=["disease","PD","Control"],alpha=alpha,cooks_filter=True,independent_filter=True,quiet=False)
    ds.summary(); results=ds.results_df.copy(); results.index.name="gene"; results=results.reset_index()
    return dds,ds,results

## 17. Ejecutar DE por tipo celular

In [ ]:
de_results={}; de_run_status=[]
if not HAS_PYDESEQ2:
    print("PyDESeq2 unavailable. Install it and rerun from this section.")
else:
    for cell_type,obj in filtered_pseudobulk_objects.items():
        counts_df=obj["counts"]; metadata_df=obj["metadata"]
        n_control=int((metadata_df["disease"]=="Control").sum()); n_pd=int((metadata_df["disease"]=="PD").sum())
        if n_control<MIN_DONORS_PER_GROUP or n_pd<MIN_DONORS_PER_GROUP:
            de_run_status.append({"cell_type":cell_type,"status":"skipped","reason":"insufficient_donors","n_control":n_control,"n_PD":n_pd}); continue
        print("\n"+"="*80); print("DE:",cell_type,"Control:",n_control,"PD:",n_pd,"Genes:",counts_df.shape[1])
        try:
            dds,ds,results=run_pydeseq2_primary(counts_df,metadata_df,ALPHA)
            de_results[cell_type]={"dds":dds,"ds":ds,"results":results}
            safe_name=re.sub(r"[^A-Za-z0-9_.-]+","_",cell_type)
            results.to_csv(NB07_RESULTS/f"DE_{safe_name}_PD_vs_Control.csv",index=False)
            de_run_status.append({"cell_type":cell_type,"status":"success","reason":"","n_control":n_control,"n_PD":n_pd})
        except Exception as exc:
            warnings.warn(f"DE failed for {cell_type}: {exc}")
            de_run_status.append({"cell_type":cell_type,"status":"failed","reason":str(exc),"n_control":n_control,"n_PD":n_pd})
        gc.collect()
de_run_status=pd.DataFrame(de_run_status)
de_run_status.to_csv(NB07_RESULTS/"GSE184950_DE_run_status.csv",index=False)
display(de_run_status)

## 18. Resumen DE

In [ ]:
de_summary_rows=[]
for cell_type,obj in de_results.items():
    res=obj["results"].copy(); sig=res["padj"].notna()&(res["padj"]<ALPHA); strong=sig&(res["log2FoldChange"].abs()>=1)
    de_summary_rows.append({"cell_type":cell_type,"n_tested":len(res),"FDR_lt_0.05":int(sig.sum()),"FDR_lt_0.05_absLFC_ge_1":int(strong.sum()),"up_PD_FDR_lt_0.05":int((sig&(res["log2FoldChange"]>0)).sum()),"down_PD_FDR_lt_0.05":int((sig&(res["log2FoldChange"]<0)).sum())})
de_summary=pd.DataFrame(de_summary_rows)
de_summary.to_csv(NB07_RESULTS/"GSE184950_celltype_DE_summary.csv",index=False)
display(de_summary.sort_values("FDR_lt_0.05",ascending=False) if len(de_summary) else de_summary)

## 19. Volcano plots por tipo celular

In [ ]:
for cell_type,obj in de_results.items():
    res=obj["results"].copy(); res["minus_log10_padj"]=-np.log10(res["padj"].clip(lower=1e-300)); sig=res["padj"].notna()&(res["padj"]<ALPHA)
    fig,ax=plt.subplots(figsize=(6,5)); ax.scatter(res.loc[~sig,"log2FoldChange"],res.loc[~sig,"minus_log10_padj"],s=4,alpha=.35,rasterized=True); ax.scatter(res.loc[sig,"log2FoldChange"],res.loc[sig,"minus_log10_padj"],s=6,alpha=.6,rasterized=True)
    ax.axvline(0,linewidth=.8); ax.axhline(-np.log10(ALPHA),linewidth=.8,linestyle="--"); ax.set_xlabel("log2 fold change (PD / Control)"); ax.set_ylabel("-log10 adjusted p-value"); ax.set_title(f"{cell_type}: PD vs Control")
    fig.tight_layout(); safe_name=re.sub(r"[^A-Za-z0-9_.-]+","_",cell_type); fig.savefig(NB07_FIGURES/f"volcano_{safe_name}.png",dpi=300,bbox_inches="tight"); plt.show(); plt.close(fig)

## 20. Firma metabólica primaria del Notebook 05

In [ ]:
PRIMARY_SIGNATURE_FALLBACK=["GPI","MDH2","ALDOC","TH","DDC","FH","NDUFS5","ALDH1A1","ACO2","PKM","IDH3G","GAPDH","G6PD","NDUFV2","IDH2"]
primary_signature=PRIMARY_SIGNATURE_FALLBACK.copy()
print("Primary signature:",primary_signature)

## 21. Resolver IDs de genes

In [ ]:
def build_symbol_lookup(adata):
    lookup={}
    for i,name in enumerate(adata.var_names.astype(str)): lookup.setdefault(name.upper(),i)
    symbol_cols=[c for c in adata.var.columns if c.lower() in {"gene_symbols","gene_symbol","symbol","gene_name","gene_names"}]
    for col in symbol_cols:
        vals=adata.var[col].astype(str)
        for i,symbol in enumerate(vals):
            if symbol and symbol.lower()!="nan": lookup.setdefault(symbol.upper(),i)
    return lookup
symbol_lookup=build_symbol_lookup(adata)
signature_mapping=[]
for gene in primary_signature:
    idx=symbol_lookup.get(gene.upper())
    signature_mapping.append({"signature_gene":gene,"found":idx is not None,"adata_var_name":str(adata.var_names[idx]) if idx is not None else None,"var_index":idx})
signature_mapping=pd.DataFrame(signature_mapping)
signature_mapping.to_csv(NB07_RESULTS/"Notebook05_signature_mapping_GSE184950.csv",index=False)
display(signature_mapping)
print("Mapped:",int(signature_mapping["found"].sum()),"/",len(signature_mapping))

## 22. Validación DE de la firma metabólica por tipo celular

In [ ]:
signature_validation_rows=[]
mapped_signature=signature_mapping.loc[signature_mapping["found"],["signature_gene","adata_var_name"]].copy()
for cell_type,obj in de_results.items():
    res=obj["results"].copy(); res["gene_upper"]=res["gene"].astype(str).str.upper()
    for _,row in mapped_signature.iterrows():
        sig_gene=row["signature_gene"]; adata_gene=str(row["adata_var_name"]); match=res.loc[res["gene_upper"]==adata_gene.upper()]
        if len(match)==0:
            signature_validation_rows.append({"cell_type":cell_type,"signature_gene":sig_gene,"adata_gene":adata_gene,"tested":False}); continue
        r=match.iloc[0]; lfc=r.get("log2FoldChange",np.nan); padj=r.get("padj",np.nan)
        signature_validation_rows.append({"cell_type":cell_type,"signature_gene":sig_gene,"adata_gene":adata_gene,"tested":True,"baseMean":r.get("baseMean",np.nan),"log2FoldChange":lfc,"lfcSE":r.get("lfcSE",np.nan),"stat":r.get("stat",np.nan),"pvalue":r.get("pvalue",np.nan),"padj":padj,"direction_PD":"Up" if lfc>0 else "Down" if lfc<0 else "No_change","FDR_lt_0.05":bool(pd.notna(padj) and padj<ALPHA)})
signature_validation=pd.DataFrame(signature_validation_rows)
signature_validation.to_csv(NB07_RESULTS/"GSE184950_Notebook05_primary_signature_validation.csv",index=False)
display(signature_validation)

## 23. Matriz de log2FC de la firma

In [ ]:
if len(signature_validation):
    lfc_matrix=signature_validation.loc[signature_validation["tested"].eq(True)].pivot_table(index="signature_gene",columns="cell_type",values="log2FoldChange",aggfunc="first")
    lfc_matrix.to_csv(NB07_RESULTS/"GSE184950_signature_log2FC_matrix.csv"); display(lfc_matrix)
    if lfc_matrix.shape[0]>0 and lfc_matrix.shape[1]>0:
        fig,ax=plt.subplots(figsize=(max(7,lfc_matrix.shape[1]*.8),max(6,lfc_matrix.shape[0]*.35)))
        im=ax.imshow(lfc_matrix.to_numpy(dtype=float),aspect="auto"); ax.set_xticks(np.arange(lfc_matrix.shape[1])); ax.set_xticklabels(lfc_matrix.columns,rotation=45,ha="right"); ax.set_yticks(np.arange(lfc_matrix.shape[0])); ax.set_yticklabels(lfc_matrix.index); ax.set_title("Notebook 05 metabolic signature — log2FC PD vs Control"); fig.colorbar(im,ax=ax,label="log2FC"); fig.tight_layout(); fig.savefig(NB07_FIGURES/"Notebook05_signature_log2FC_heatmap.png",dpi=300,bbox_inches="tight"); plt.show(); plt.close(fig)

## 24. Concordancia con bulk

In [ ]:
BULK_DIRECTION={g:"Down" for g in PRIMARY_SIGNATURE_FALLBACK}
if len(signature_validation):
    signature_validation["bulk_expected_direction"] = signature_validation["signature_gene"].map(BULK_DIRECTION)
    signature_validation["direction_concordant"] = signature_validation["direction_PD"] == signature_validation["bulk_expected_direction"]
    concordance_summary = (
        signature_validation.loc[signature_validation["tested"].eq(True)]
        .groupby("cell_type", observed=True)
        .agg(
            n_signature_genes_tested=("signature_gene", "nunique"),
            n_concordant=("direction_concordant", "sum"),
            n_FDR_lt_005=("FDR_lt_0.05", "sum"),
        )
        .reset_index()
    )
    concordance_summary["fraction_concordant"] = (
        concordance_summary["n_concordant"] /
        concordance_summary["n_signature_genes_tested"]
    )
    concordance_summary.to_csv(
        NB07_RESULTS / "GSE184950_signature_direction_concordance.csv",
        index=False,
    )
    display(concordance_summary.sort_values("fraction_concordant", ascending=False))

## 25. Representación para análisis secundario PDD vs Control

In [ ]:
secondary_rep=rep_pivot.copy(); secondary_rep["eligible_PDD_vs_Control"]=(secondary_rep["Control"]>=MIN_DONORS_PER_GROUP)&(secondary_rep["PDD"]>=MIN_DONORS_PER_GROUP)
secondary_celltypes=secondary_rep.index[secondary_rep["eligible_PDD_vs_Control"]].astype(str).tolist(); secondary_rep.to_csv(NB07_RESULTS/"GSE184950_PDD_secondary_representation.csv")
print("Cell types eligible for secondary PDD vs Control analysis:",len(secondary_celltypes))
for ct in secondary_celltypes: print(f"- {ct}: Control={secondary_rep.loc[ct,'Control']}, PDD={secondary_rep.loc[ct,'PDD']}")

## 26. Resumen conservador por tipo celular

In [ ]:
interpretation_rows=[]
for _,row in de_summary.iterrows():
    cell_type=row["cell_type"]; sig_sub=signature_validation.loc[signature_validation["cell_type"].eq(cell_type)&signature_validation["tested"].eq(True)] if len(signature_validation) else pd.DataFrame()
    interpretation_rows.append({"cell_type":cell_type,"n_DE_FDR_lt_0.05":int(row["FDR_lt_0.05"]),"n_signature_tested":int(len(sig_sub)),"n_signature_FDR_lt_0.05":int(sig_sub["FDR_lt_0.05"].sum()) if len(sig_sub) else 0,"n_signature_down_PD":int(sig_sub["direction_PD"].eq("Down").sum()) if len(sig_sub) else 0,"n_signature_up_PD":int(sig_sub["direction_PD"].eq("Up").sum()) if len(sig_sub) else 0})
interpretation_table=pd.DataFrame(interpretation_rows); interpretation_table.to_csv(NB07_RESULTS/"GSE184950_celltype_interpretation_summary.csv",index=False); display(interpretation_table)

## 27. Exportar tabla maestra para Notebook 08

In [ ]:
cross_dataset_export=signature_validation.copy() if len(signature_validation) else pd.DataFrame()
if len(cross_dataset_export):
    cross_dataset_export["bulk_dataset"]="GSE136666"; cross_dataset_export["validation_dataset"]="GSE184950"; cross_dataset_export["bulk_tissue"]="Substantia nigra"; cross_dataset_export["validation_tissue"]="Substantia nigra"; cross_dataset_export["validation_modality"]="snRNA-seq pseudobulk"; cross_dataset_export["biological_replicate"]="donor"
    cross_dataset_export.to_csv(NB07_RESULTS/"Notebook08_cross_dataset_input.csv",index=False)
display(cross_dataset_export.head())

## 28. Manifest de Notebook 07

In [ ]:
manifest07={"dataset":"GSE184950","modality":"single-nucleus RNA-seq","tissue":"substantia nigra","input_h5ad":str(H5AD_PATH),"biological_replicate":"donor","aggregation_unit":"donor x cell type","raw_count_layer":"counts","primary_contrast":"PD vs Control","secondary_contrast":"PDD vs Control","min_nuclei_per_pseudobulk":MIN_NUCLEI_PER_PSEUDOBULK,"min_donors_per_group":MIN_DONORS_PER_GROUP,"gene_filter":f"count >= {MIN_COUNT} in >= {MIN_SAMPLES_WITH_COUNT} pseudobulk samples","primary_model":"~ disease","celltype_labels":"cell_type_provisional","n_eligible_celltypes":int(len(eligible_celltypes)),"eligible_celltypes":eligible_celltypes,"n_successful_DE_celltypes":int(len(de_results)),"primary_signature_genes":primary_signature,"next_notebook":"08_cross_dataset_validation.ipynb","interpretation_warning":"Transcript abundance does not directly measure metabolic flux or enzyme activity."}
manifest07_path=NB07_RESULTS/"notebook07_manifest.json"
with open(manifest07_path,"w",encoding="utf-8") as f: json.dump(manifest07,f,indent=2,ensure_ascii=False)
print(json.dumps(manifest07,indent=2,ensure_ascii=False))

## 29. Checkpoint final

In [ ]:
checks07={"Input H5AD loaded":adata is not None,"Raw counts available":"counts" in adata.layers,"Donor IDs available":"donor_id_clean" in adata.obs.columns,"Disease labels available":"disease_clean" in adata.obs.columns,"Cell-type labels available":"cell_type_clean" in adata.obs.columns,"At least one eligible primary cell type":len(eligible_celltypes)>0,"Pseudobulk objects generated":len(pseudobulk_objects)>0,"Pseudobulk summary exported":(NB07_RESULTS/"GSE184950_pseudobulk_summary.csv").exists(),"Signature mapping generated":len(signature_mapping)>0,"Manifest exported":manifest07_path.exists()}
if HAS_PYDESEQ2:
    checks07["At least one successful PyDESeq2 analysis"]=len(de_results)>0
    checks07["Cross-dataset validation table exported"]=(NB07_RESULTS/"Notebook08_cross_dataset_input.csv").exists()
checkpoint07=pd.Series(checks07,name="passed").to_frame(); checkpoint07["status"]=checkpoint07["passed"].map({True:"PASS",False:"FAIL"}); display(checkpoint07); checkpoint07.to_csv(NB07_RESULTS/"notebook07_final_checkpoint.csv")
failed=checkpoint07.loc[~checkpoint07["passed"]]
if len(failed): print("\nSome Notebook 07 checks failed:"); display(failed)
else: print("\n✓ All Notebook 07 checks passed.")
print("\nNext: 08_cross_dataset_validation.ipynb")

## 30. Interpretación científica

1. Un resultado significativo en pseudobulk representa evidencia a nivel de **donantes**, no de núcleos individuales.
2. La falta de significancia con pocos donantes puede reflejar baja potencia.
3. Concordancia bulk/snRNA-seq fortalece robustez, pero no prueba causalidad.
4. Expresión transcriptómica no equivale a flujo metabólico ni actividad enzimática.
5. Los tipos celulares siguen siendo provisionales hasta contrastarlos con anotaciones publicadas o evidencia robusta de marcadores.
6. Notebook 08 integrará resultados entre datasets **sin concatenar matrices bulk y snRNA-seq**.